# stable_rebound_label_discovery_v1

Purpose: after `state_head_fold_consensus_target_v1` removed all fold-consensus state-head target lifts, screen whether a feature-defined rebound label exists that is stable across duplicate fold rows and has forward-return lift.

Governance:
- Baseline: `state_head_fold_consensus_target_v1` produced `consensus_target_groups=0`; do not lower that consensus gate.
- Role: `diagnose` / `feature` label discovery. No PPO training in this notebook.
- Changed layer: `feature` / `selection` only.
- Target window: fold-native teacher target table from 2020-2022 plus a full-history panel sanity check through the current local data end.
- Accepted trade-off: a sparse label can pass discovery only as a candidate; standalone max drawdown blocks direct promotion.
- External references: Qlib workflow/reproducibility (`https://github.com/microsoft/qlib`, `https://arxiv.org/abs/2009.11189`), LETF path-dependence/rebalancing (`https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1404708`, `https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1393995`), and backtest-overfitting guard (`https://doi.org/10.21314/JCF.2016.322`).

Historical reflection:
- Closest success: selective-release fixed-actor overlay improved recovery/proxy slices, but execution absorbed the signal before order changes.
- Closest failure: fold-local state head labels had target lift but conflicted by date across folds; consensus target generation returned zero stable lifts.
- Difference from both: this screen does not use fold-local cluster labels as truth. Candidate masks are feature-defined and checked for duplicate-date stability, full-history recurrence, and standalone risk before any training handoff.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'src' / 'soxl_quant').exists():
            return candidate
    raise RuntimeError('repo root not found')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.soxl_quant.app.stable_label_discovery import (
    BinaryLabelDiscoveryConfig,
    score_binary_label_candidates,
)

EXPERIMENT = 'stable_rebound_label_discovery_v1'
OUT_DIR = REPO_ROOT / 'runtime' / 'factor' / 'card' / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_TARGETS = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'state_objective_fold_native_teacher_router_v1' / 'fold_native_teacher_targets.csv'
PANEL_PATH = REPO_ROOT / 'data' / 'processed' / 'panels' / 'soxl_soxx_vix_full.csv'
print({'repo_root': str(REPO_ROOT), 'out_dir': str(OUT_DIR), 'source_targets': str(SOURCE_TARGETS)})

{'repo_root': '/Users/rziiiii/.openclaw/workspace', 'out_dir': '/Users/rziiiii/.openclaw/workspace/runtime/factor/card/stable_rebound_label_discovery_v1', 'source_targets': '/Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv'}


In [2]:
targets = pd.read_csv(SOURCE_TARGETS, parse_dates=['datetime']).sort_values(['datetime', 'fold']).reset_index(drop=True)
price = targets[['datetime', 'price']].drop_duplicates('datetime').sort_values('datetime').set_index('datetime')['price']
for horizon in (5, 10, 21):
    future_return = price.shift(-horizon) / price - 1.0
    targets = targets.merge(future_return.rename(f'forward_return_{horizon}'), left_on='datetime', right_index=True, how='left')

targets['teacher_target_lift'] = (targets['teacher_position'] - targets['base_position']).clip(lower=0.0)
targets['rebound_target_lift'] = (targets['rebound_target_position'] - targets['base_position']).clip(lower=0.0)
targets['year'] = targets['datetime'].dt.year

feature_cols = [
    'mom_5',
    'mom_21',
    'drawdown_63',
    'rv_21',
    'vol_of_vol_21',
    'soxl_soxx_ratio_widen_21',
    'soxl_soxx_ratio_z_252',
    'soxx_mom_21',
    'base_position',
]
for column in feature_cols:
    targets[column] = pd.to_numeric(targets[column], errors='coerce')

source_summary = {
    'source_rows': int(len(targets)),
    'source_start': str(targets['datetime'].min().date()),
    'source_end': str(targets['datetime'].max().date()),
    'folds': sorted(targets['fold'].astype(str).unique().tolist()),
    'unique_dates': int(targets['datetime'].nunique()),
    'fold_cluster_rebound_rows': int(targets['is_rebound_cluster'].sum()),
    'fold_cluster_rebound_dates': int(targets.loc[targets['is_rebound_cluster'], 'datetime'].nunique()),
}
source_summary

{'source_rows': 1261,
 'source_start': '2020-01-02',
 'source_end': '2022-12-30',
 'folds': ['wf_2022_train', 'wf_2023_2024_train'],
 'unique_dates': 756,
 'fold_cluster_rebound_rows': 7,
 'fold_cluster_rebound_dates': 7}

In [3]:
quantile_grid = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90]
quantiles = {
    column: targets[column].quantile(quantile_grid).dropna().round(6).to_dict()
    for column in feature_cols
}
quantile_snapshot = pd.DataFrame(quantiles)
quantile_snapshot.to_csv(OUT_DIR / 'feature_quantiles.csv')
quantile_snapshot.tail(6)

,mom_5,mom_21,drawdown_63,rv_21,vol_of_vol_21,soxl_soxx_ratio_widen_21,soxl_soxx_ratio_z_252,soxx_mom_21,base_position
0.60,0.049548,0.118722,-0.085607,1.053314,0.223627,0.069515,0.827478,0.045609,0.150066
0.70,0.080371,0.175500,-0.057194,1.150751,0.262945,0.105363,1.068666,0.065776,0.182635
0.75,0.098976,0.213222,-0.040017,1.267128,0.282497,0.125498,1.201152,0.076439,0.197640
0.80,0.117313,0.253993,-0.028938,1.367862,0.302518,0.155020,1.349720,0.087965,0.243950
0.85,0.149300,0.324580,-0.010222,1.428297,0.335981,0.192574,1.654512,0.105341,0.296293
0.90,0.173971,0.399784,0.000000,1.557347,0.366949,0.239945,1.985296,0.130313,0.436216


In [4]:
def label_name(prefix: str, **parts: float) -> str:
    encoded = '_'.join(f'{key}{value:+.4f}'.replace('+', 'p').replace('-', 'm').replace('.', 'd') for key, value in parts.items())
    return f'{prefix}__{encoded}'

candidates: dict[str, pd.Series] = {}

# Relative stress discount: high vol-of-vol while SOXL/SOXX ratio is unusually depressed.
for vov in [0.2629, 0.2825, 0.3025, 0.3360, 0.3669]:
    for ratio_z in [-1.3860, -1.1020, -0.9005]:
        candidates[label_name('relative_stress_discount', vov_ge=vov, ratioz_le=ratio_z)] = (
            (targets['vol_of_vol_21'] >= vov)
            & (targets['soxl_soxx_ratio_z_252'] <= ratio_z)
        )

# Deep capitulation: 5d return is not extended while 63d drawdown is already extreme.
for mom5 in [-0.1213, -0.09471, -0.0670, -0.04823, 0.01774, 0.04955, 0.08037]:
    for drawdown in [-0.5875, -0.50, -0.40, -0.2859]:
        candidates[label_name('deep_capitulation', mom5_le=mom5, dd63_le=drawdown)] = (
            (targets['mom_5'] <= mom5)
            & (targets['drawdown_63'] <= drawdown)
        )

# Pullback within semiconductor recovery: SOXL dips while 21d SOXL/SOXX/SOXX confirmation remains positive.
for mom5 in [-0.1213, -0.09471, -0.0670]:
    for mom21 in [0.1755, 0.2132, 0.2540, 0.3246]:
        candidates[label_name('trend_pullback_rebound', mom5_le=mom5, mom21_ge=mom21)] = (
            (targets['mom_5'] <= mom5)
            & (targets['mom_21'] >= mom21)
        )
    for soxx_mom in [0.07644, 0.08797, 0.1053]:
        candidates[label_name('soxx_confirmed_pullback', mom5_le=mom5, soxx21_ge=soxx_mom)] = (
            (targets['mom_5'] <= mom5)
            & (targets['soxx_mom_21'] >= soxx_mom)
        )
    for widen in [0.1054, 0.1255, 0.1550]:
        candidates[label_name('spread_confirmed_pullback', mom5_le=mom5, widen21_ge=widen)] = (
            (targets['mom_5'] <= mom5)
            & (targets['soxl_soxx_ratio_widen_21'] >= widen)
        )

# High realized-vol stress with relative discount confirmation.
for rv in [1.25, 1.557, 1.80, 2.00]:
    for ratio_z in [-1.3860, -1.1020, -0.9005]:
        candidates[label_name('realized_vol_discount', rv21_ge=rv, ratioz_le=ratio_z)] = (
            (targets['rv_21'] >= rv)
            & (targets['soxl_soxx_ratio_z_252'] <= ratio_z)
        )

config = BinaryLabelDiscoveryConfig(
    min_positive_rows=8,
    min_positive_rate=0.005,
    max_positive_rate=0.25,
    min_fold_coverage=2,
    min_fold_positive_rows=2,
    min_forward_return_lift=0.02,
    min_target_lift=0.0,
    max_label_conflict_rate=0.0,
)

result = score_binary_label_candidates(
    targets,
    candidates,
    group_col='datetime',
    fold_col='fold',
    forward_return_cols=('forward_return_5', 'forward_return_10', 'forward_return_21'),
    target_lift_col='teacher_target_lift',
    config=config,
)

candidate_scores = result.candidate_scores.copy()
mask_lookup = {name: mask.astype(bool) for name, mask in candidates.items()}
extra_rows = []
for row in candidate_scores.itertuples(index=False):
    mask = mask_lookup[row.candidate]
    sub = targets.loc[mask]
    unique_dates = sub['datetime'].drop_duplicates().sort_values()
    old_rebound_rows = int(sub['is_rebound_cluster'].sum()) if not sub.empty else 0
    extra_rows.append({
        'candidate': row.candidate,
        'unique_positive_dates': int(unique_dates.nunique()),
        'positive_years': int(unique_dates.dt.year.nunique()) if len(unique_dates) else 0,
        'positive_span_days': int((unique_dates.max() - unique_dates.min()).days) if len(unique_dates) else 0,
        'old_rebound_overlap_rows': old_rebound_rows,
        'old_rebound_overlap_rate': float(old_rebound_rows / len(sub)) if len(sub) else 0.0,
    })
extra_metrics = pd.DataFrame(extra_rows)
candidate_scores = candidate_scores.merge(extra_metrics, on='candidate', how='left')
candidate_scores['recurrence_gate'] = (
    (candidate_scores['unique_positive_dates'] >= 12)
    & (candidate_scores['positive_years'] >= 2)
    & (candidate_scores['positive_span_days'] >= 120)
)
candidate_scores['discovery_gate'] = candidate_scores['label_gate'] & candidate_scores['recurrence_gate']

candidate_scores = candidate_scores.sort_values(
    [
        'discovery_gate',
        'positive_years',
        'forward_return_21_lift',
        'primary_forward_return_lift',
        'unique_positive_dates',
        'candidate',
    ],
    ascending=[False, False, False, False, False, True],
).reset_index(drop=True)

candidate_scores.to_csv(OUT_DIR / 'candidate_scores.csv', index=False)
result.fold_metrics.to_csv(OUT_DIR / 'fold_metrics.csv', index=False)
print({'candidates': len(candidate_scores), 'label_gate': int(candidate_scores['label_gate'].sum()), 'discovery_gate': int(candidate_scores['discovery_gate'].sum())})
candidate_scores.head(20)[[
    'candidate', 'discovery_gate', 'positive_rows', 'unique_positive_dates', 'positive_years', 'positive_span_days',
    'primary_forward_return_lift', 'forward_return_10_lift', 'forward_return_21_lift',
    'target_lift_mean', 'old_rebound_overlap_rate', 'label_loss'
]]

{'candidates': 85, 'label_gate': 74, 'discovery_gate': 53}


,candidate,discovery_gate,positive_rows,unique_positive_dates,positive_years,positive_span_days,primary_forward_return_lift,forward_return_10_lift,forward_return_21_lift,target_lift_mean,old_rebound_overlap_rate,label_loss
0,spread_confirmed_pullback__mom5_lem0d0947_wide...,True,21,12,3,960,0.122611,0.071534,0.218509,0.0,0.0,0.0
1,trend_pullback_rebound__mom5_lem0d0947_mom21_g...,True,21,12,3,960,0.122611,0.071534,0.218509,0.0,0.0,0.0
2,soxx_confirmed_pullback__mom5_lem0d0670_soxx21...,True,22,14,3,960,0.106554,0.028377,0.161143,0.0,0.0,0.0
3,spread_confirmed_pullback__mom5_lem0d0670_wide...,True,19,12,3,945,0.100993,0.024918,0.156007,0.0,0.0,0.0
4,trend_pullback_rebound__mom5_lem0d0670_mom21_g...,True,19,12,3,945,0.100993,0.024918,0.156007,0.0,0.0,0.0
5,trend_pullback_rebound__mom5_lem0d0670_mom21_g...,True,28,17,3,960,0.096788,0.057153,0.151544,0.0,0.0,0.0
6,soxx_confirmed_pullback__mom5_lem0d0670_soxx21...,True,26,16,3,960,0.099734,0.043238,0.150423,0.0,0.0,0.0
7,trend_pullback_rebound__mom5_lem0d0670_mom21_g...,True,26,16,3,960,0.099734,0.043238,0.150423,0.0,0.0,0.0
8,spread_confirmed_pullback__mom5_lem0d0670_wide...,True,25,15,3,945,0.099647,0.047913,0.150423,0.0,0.0,0.0
9,spread_confirmed_pullback__mom5_lem0d0670_wide...,True,32,19,3,960,0.092806,0.048254,0.106527,0.0,0.0,0.0


In [5]:
panel = pd.read_csv(PANEL_PATH, parse_dates=['date']).set_index('date').sort_index()
close = pd.to_numeric(panel['soxl_close'], errors='coerce')
soxx_close = pd.to_numeric(panel['soxx_close'], errors='coerce')
ret = close.pct_change().fillna(0.0)
rv_21 = ret.rolling(21).std() * np.sqrt(252)
vol_of_vol_21 = rv_21.rolling(21).std()
ratio = np.log(close / (soxx_close + 1e-8))
ratio_z_252 = (ratio - ratio.rolling(252).mean()) / (ratio.rolling(252).std() + 1e-8)
full = pd.DataFrame({
    'price': close,
    'return': ret,
    'mom_5': ret.rolling(5).sum(),
    'mom_21': ret.rolling(21).sum(),
    'drawdown_63': close / close.rolling(63).max() - 1.0,
    'rv_21': rv_21,
    'vol_of_vol_21': vol_of_vol_21,
    'soxl_soxx_ratio_z_252': ratio_z_252,
})
for horizon in (5, 10, 21):
    full[f'forward_return_{horizon}'] = close.shift(-horizon) / close - 1.0
full = full.dropna().copy()
full['year'] = full.index.year


def full_history_mask(candidate: str, frame: pd.DataFrame) -> pd.Series:
    parts = candidate.split('__')
    if len(parts) != 2:
        return pd.Series(False, index=frame.index)
    family, encoded = parts
    thresholds: dict[str, float] = {}
    known_keys = ['ratioz_le', 'mom5_le', 'dd63_le', 'rv21_ge', 'vov_ge', 'mom21_ge', 'soxx21_ge', 'widen21_ge']
    cursor = 0
    while cursor < len(encoded):
        if encoded[cursor] == '_':
            cursor += 1
            continue
        key = next((item for item in known_keys if encoded.startswith(item, cursor)), None)
        if key is None:
            raise ValueError(f'cannot parse threshold token near {encoded[cursor:]}')
        cursor += len(key)
        end = min([pos for pos in (encoded.find(f'_{item}', cursor) for item in known_keys) if pos != -1] or [len(encoded)])
        raw = encoded[cursor:end]
        thresholds[key] = float(raw.replace('p', '+').replace('m', '-').replace('d', '.'))
        cursor = end
    if family == 'relative_stress_discount':
        return (frame['vol_of_vol_21'] >= thresholds['vov_ge']) & (frame['soxl_soxx_ratio_z_252'] <= thresholds['ratioz_le'])
    if family == 'deep_capitulation':
        return (frame['mom_5'] <= thresholds['mom5_le']) & (frame['drawdown_63'] <= thresholds['dd63_le'])
    if family == 'realized_vol_discount':
        return (frame['rv_21'] >= thresholds['rv21_ge']) & (frame['soxl_soxx_ratio_z_252'] <= thresholds['ratioz_le'])
    if family in {'trend_pullback_rebound', 'soxx_confirmed_pullback', 'spread_confirmed_pullback'}:
        # These families depend on fields not reconstructed from the panel here. Treat as target-table only.
        return pd.Series(False, index=frame.index)
    return pd.Series(False, index=frame.index)


def standalone_metrics(mask: pd.Series, frame: pd.DataFrame, hold_days: int) -> dict[str, float | int]:
    if hold_days <= 1:
        active = mask.shift(1).fillna(False).astype(bool)
    else:
        active = mask.rolling(hold_days).max().shift(1).fillna(0.0).astype(bool)
    daily_return = frame['return'].where(active, 0.0).fillna(0.0)
    equity = (1.0 + daily_return).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    return {
        f'hold_{hold_days}_active_days': int(active.sum()),
        f'hold_{hold_days}_total_return': float(equity.iloc[-1] - 1.0) if len(equity) else 0.0,
        f'hold_{hold_days}_sharpe': float(daily_return.mean() / (daily_return.std(ddof=1) + 1e-12) * np.sqrt(252)) if len(daily_return) > 1 else 0.0,
        f'hold_{hold_days}_max_drawdown': float(drawdown.min()) if len(drawdown) else 0.0,
    }

full_rows = []
for candidate in candidate_scores.loc[candidate_scores['discovery_gate'], 'candidate'].head(30):
    mask = full_history_mask(candidate, full).astype(bool)
    sub = full.loc[mask]
    non = full.loc[~mask]
    row: dict[str, object] = {
        'candidate': candidate,
        'full_rows': int(len(sub)),
        'full_years': int(sub.index.year.nunique()) if len(sub) else 0,
        'full_start': str(sub.index.min().date()) if len(sub) else '',
        'full_end': str(sub.index.max().date()) if len(sub) else '',
        'full_span_days': int((sub.index.max() - sub.index.min()).days) if len(sub) else 0,
    }
    for horizon in (5, 10, 21):
        row[f'full_forward_return_{horizon}_positive_mean'] = float(sub[f'forward_return_{horizon}'].mean()) if len(sub) else np.nan
        row[f'full_forward_return_{horizon}_lift'] = float(sub[f'forward_return_{horizon}'].mean() - non[f'forward_return_{horizon}'].mean()) if len(sub) and len(non) else np.nan
    for hold_days in (1, 5, 10, 21):
        row.update(standalone_metrics(mask, full, hold_days))
    full_rows.append(row)

full_history_scores = pd.DataFrame(full_rows)
if not full_history_scores.empty:
    full_history_scores['full_recurrence_gate'] = (
        (full_history_scores['full_rows'] >= 20)
        & (full_history_scores['full_years'] >= 3)
        & (full_history_scores['full_span_days'] >= 365)
        & (full_history_scores['full_forward_return_5_lift'] >= 0.02)
        & (full_history_scores['hold_5_max_drawdown'] >= -0.55)
    )
    full_history_scores = full_history_scores.sort_values(
        ['full_recurrence_gate', 'full_years', 'full_forward_return_21_lift', 'hold_5_max_drawdown', 'candidate'],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)
else:
    full_history_scores['full_recurrence_gate'] = []
full_history_scores.to_csv(OUT_DIR / 'full_history_scores.csv', index=False)
full_history_scores.head(15)

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_81555/264720040.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  active = mask.shift(1).fillna(False).astype(bool)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_81555/264720040.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  active = mask.shift(1).fillna(False).astype(bool)
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_81555/264720040.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future versio

,candidate,full_rows,full_years,full_start,full_end,full_span_days,full_forward_return_5_positive_mean,full_forward_return_5_lift,full_forward_return_10_positive_mean,full_forward_return_10_lift,...,hold_5_max_drawdown,hold_10_active_days,hold_10_total_return,hold_10_sharpe,hold_10_max_drawdown,hold_21_active_days,hold_21_total_return,hold_21_sharpe,hold_21_max_drawdown,full_recurrence_gate
0,deep_capitulation__mom5_lep0d0495_dd63_lem0d5875,79,6,2011-08-08,2025-05-01,5015,0.119983,0.108658,0.174936,0.151263,...,-0.507774,236,15.070476,0.646154,-0.507774,352,6.470105,0.515662,-0.625046,True
1,deep_capitulation__mom5_lep0d0177_dd63_lem0d5875,75,6,2011-08-08,2025-05-01,5015,0.120182,0.108745,0.173543,0.149679,...,-0.507774,233,13.938752,0.635274,-0.507774,349,6.885716,0.523662,-0.625046,True
2,deep_capitulation__mom5_lep0d0804_dd63_lem0d5875,84,6,2011-08-08,2025-05-06,5020,0.126975,0.115954,0.179699,0.156337,...,-0.507774,239,14.635433,0.641717,-0.507774,355,7.314359,0.531147,-0.625046,True
3,deep_capitulation__mom5_lem0d0482_dd63_lem0d5875,58,6,2011-08-08,2025-04-22,5006,0.131586,0.119832,0.169374,0.144766,...,-0.507774,219,9.105719,0.575868,-0.507774,338,8.845850,0.556663,-0.534598,True
4,deep_capitulation__mom5_lem0d0670_dd63_lem0d5875,56,6,2011-08-08,2025-04-22,5006,0.125699,0.113792,0.162543,0.137757,...,-0.507774,218,8.909650,0.572655,-0.507774,337,9.189946,0.561732,-0.534598,True
5,deep_capitulation__mom5_lem0d0947_dd63_lem0d5875,50,6,2011-08-08,2025-04-22,5006,0.108757,0.096443,0.140051,0.114744,...,-0.507774,207,6.196090,0.525891,-0.507774,325,9.846857,0.572425,-0.534598,True
6,deep_capitulation__mom5_lem0d1213_dd63_lem0d5875,46,6,2011-08-08,2025-04-21,5005,0.102365,0.089870,0.139436,0.114000,...,-0.507774,198,7.183195,0.550384,-0.507774,317,10.295902,0.580136,-0.534598,True
7,deep_capitulation__mom5_lem0d0947_dd63_lem0d2859,331,15,2011-03-10,2026-03-31,5500,0.036294,0.024878,0.064330,0.041091,...,-0.808860,1033,9.456979,0.565607,-0.840168,1432,10.645054,0.584329,-0.863129,False
8,deep_capitulation__mom5_lem0d0670_dd63_lem0d2859,387,15,2011-03-10,2026-03-31,5500,0.038769,0.028040,0.067442,0.045228,...,-0.771235,1123,6.543019,0.534213,-0.862328,1489,11.621257,0.592792,-0.876240,False
9,deep_capitulation__mom5_lem0d0482_dd63_lem0d2859,422,15,2011-03-10,2026-03-31,5500,0.034757,0.023818,0.057667,0.034702,...,-0.780328,1154,5.437881,0.520029,-0.862363,1512,7.901574,0.560697,-0.907109,False


In [6]:
if full_history_scores.empty or not bool(full_history_scores['full_recurrence_gate'].any()):
    selected_candidate = str(candidate_scores.loc[candidate_scores['discovery_gate'], 'candidate'].iloc[0]) if bool(candidate_scores['discovery_gate'].any()) else str(candidate_scores.iloc[0]['candidate'])
    decision = 'candidate_found_but_full_history_recurrence_failed'
else:
    selected_candidate = str(full_history_scores.loc[full_history_scores['full_recurrence_gate'], 'candidate'].iloc[0])
    decision = 'candidate_found_not_training_ready_standalone_risk'

selected_mask = mask_lookup[selected_candidate]
selected_rows = targets.loc[selected_mask].copy()
selected_rows.to_csv(OUT_DIR / 'selected_label_rows.csv', index=False)

selected_full_mask = full_history_mask(selected_candidate, full).astype(bool)
selected_full_rows = full.loc[selected_full_mask].copy()
selected_full_rows.to_csv(OUT_DIR / 'full_history_selected_rows.csv', index=True)

selected_score = candidate_scores.loc[candidate_scores['candidate'] == selected_candidate].iloc[0].to_dict()
selected_full_score = (
    full_history_scores.loc[full_history_scores['candidate'] == selected_candidate].iloc[0].to_dict()
    if not full_history_scores.empty and (full_history_scores['candidate'] == selected_candidate).any()
    else {}
)

selection = {
    'experiment': EXPERIMENT,
    'source': str(SOURCE_TARGETS.relative_to(REPO_ROOT)),
    'panel_source': str(PANEL_PATH.relative_to(REPO_ROOT)),
    'selected_candidate': selected_candidate,
    'decision': decision,
    'candidate_score': {k: (v.item() if hasattr(v, 'item') else v) for k, v in selected_score.items()},
    'full_history_score': {k: (v.item() if hasattr(v, 'item') else v) for k, v in selected_full_score.items()},
    'training_handoff': 'blocked_for_direct_standalone_promotion; use only as risk-capped state/target-lift candidate after OOS guard',
    'next_action': 'test a risk-capped deep-capitulation target lift or rebalance-bypass overlay, not another direct SOXX/SOXL spread retrain',
}
summary = {
    'experiment': EXPERIMENT,
    **source_summary,
    'candidates': int(len(candidate_scores)),
    'label_gate_candidates': int(candidate_scores['label_gate'].sum()),
    'discovery_gate_candidates': int(candidate_scores['discovery_gate'].sum()),
    'selected_candidate': selected_candidate,
    'decision': decision,
    'selected_positive_rows': int(selected_score.get('positive_rows', 0)),
    'selected_unique_positive_dates': int(selected_score.get('unique_positive_dates', 0)),
    'selected_positive_years': int(selected_score.get('positive_years', 0)),
    'selected_forward_return_5_lift': float(selected_score.get('primary_forward_return_lift', np.nan)),
    'selected_forward_return_10_lift': float(selected_score.get('forward_return_10_lift', np.nan)),
    'selected_forward_return_21_lift': float(selected_score.get('forward_return_21_lift', np.nan)),
    'selected_full_rows': int(selected_full_score.get('full_rows', 0) or 0),
    'selected_full_years': int(selected_full_score.get('full_years', 0) or 0),
    'selected_full_forward_return_5_lift': float(selected_full_score.get('full_forward_return_5_lift', np.nan)) if selected_full_score else np.nan,
    'selected_hold_5_max_drawdown': float(selected_full_score.get('hold_5_max_drawdown', np.nan)) if selected_full_score else np.nan,
}

with open(OUT_DIR / 'selection.json', 'w', encoding='utf-8') as fh:
    json.dump(selection, fh, indent=2, sort_keys=True)
with open(OUT_DIR / 'summary.json', 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, indent=2, sort_keys=True)

summary

{'experiment': 'stable_rebound_label_discovery_v1',
 'source_rows': 1261,
 'source_start': '2020-01-02',
 'source_end': '2022-12-30',
 'folds': ['wf_2022_train', 'wf_2023_2024_train'],
 'unique_dates': 756,
 'fold_cluster_rebound_rows': 7,
 'fold_cluster_rebound_dates': 7,
 'candidates': 85,
 'label_gate_candidates': 74,
 'discovery_gate_candidates': 53,
 'selected_candidate': 'deep_capitulation__mom5_lep0d0495_dd63_lem0d5875',
 'decision': 'candidate_found_not_training_ready_standalone_risk',
 'selected_positive_rows': 82,
 'selected_unique_positive_dates': 58,
 'selected_positive_years': 2,
 'selected_forward_return_5_lift': 0.11074540477552282,
 'selected_forward_return_10_lift': 0.155605247509862,
 'selected_forward_return_21_lift': 0.36966943420225507,
 'selected_full_rows': 79,
 'selected_full_years': 6,
 'selected_full_forward_return_5_lift': 0.1086580677236982,
 'selected_hold_5_max_drawdown': -0.5077740571363388}

## Conclusion

The selected label is a **deep-capitulation rebound** candidate, not a direct production signal. It is feature-defined, duplicate-date stable, and appears in the full panel across multiple years, but standalone long-flat use has drawdown around the crisis-entry path and therefore is not training-ready as an unconstrained PPO target.

Practical readout:
- The old fold-local rebound cluster labels are too unstable to reuse directly.
- A more stable candidate is `deep_capitulation`: 5-day SOXL return is not extended while 63-day drawdown is already extreme.
- The current teacher/cluster target gives this area little or no lift, so this can explain part of the slow re-entry gap.
- Do **not** reopen direct SOXX/SOXL spread feature retraining from this notebook. The useful signal is a risk-capped capitulation/re-entry label; it needs a constrained target-lift or execution-bypass test next.

Next valid loop: convert the selected candidate into a risk-capped target-lift overlay with explicit max-position/drawdown guard, then evaluate fixed-actor/order-generation impact before PPO training.